## Step 1: Setting up the environment

I am using google colab kernel for this notebook

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np

# Confirm versions 
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# We will use CPU throughout — no GPU needed to understand the architecture
device = torch.device('cpu')
print("Using device:", device)

PyTorch version: 2.10.0+cpu
CUDA available: False
Using device: cpu


In [ ]:
d_model     = 512
num_heads   = 8
d_ff        = 2048
num_layers  = 6
dropout     = 0.1
max_seq_len = 100
vocab_size  = 1000

## Step 2 : Input Embedding Class

**InputEmbedding: converts integer token IDs into 512-dimensional dense vectors and scales them by sqrt(d_model) so they are on the same magnitude as positional encodings.**

In [3]:
class InputEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)   # nn.Embedding is a lookup table 
        self.d_model = d_model             # The self. prefix means it belongs to this object and will be accessible in the forward method.

    def forward(self, x):   # Every nn.Module must have a forward method .This defines what happens to the data when it passes through this component. 
        # x shape coming in : (batch_size, seq_len)  — integer token IDs e.g (2,5) : 2 sentences, each with 5 tokens
        # x shape going out : (batch_size, seq_len, d_model) e.g (2,5,512) : each token is now represented by a 512-dimensional vector
        return self.embedding(x) * math.sqrt(self.d_model)  # the math.sqrt(d_model) scaling is multiplied to the embeddings to scale them up. This is followed from the original transformer paper.

In [4]:
# Test InputEmbedding
embed = InputEmbedding(vocab_size, d_model)

# Fake input: batch of 2 sentences, each 5 tokens long
# Each number is a token ID (integer between 0 and vocab_size-1)
dummy_tokens = torch.tensor([
    [4, 27, 103, 56, 8],   # sentence 1
    [9, 41, 7,  200, 3],   # sentence 2
])

print("Input shape  :", dummy_tokens.shape)   # (2, 5)

output = embed(dummy_tokens)
print("Output shape :", output.shape)          # (2, 5, 512)
print("Sample vector (first token, first sentence):", output[0][0][:5])

Input shape  : torch.Size([2, 5])
Output shape : torch.Size([2, 5, 512])
Sample vector (first token, first sentence): tensor([ 21.2542, -15.1201,  23.6416, -26.3336,  -8.1797],
       grad_fn=<SliceBackward0>)


## Step 3 : Positional Encoding

In [5]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len , dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_seq_len,d_model)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)  # (max_seq_len, 1)
        div_term = torch.exp(
           torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
             )
        pe[:, 0::2] = torch.sin(position * div_term)  # even indices
        pe[:, 1::2] = torch.cos(position * div_term)  # odd
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)  # Register as buffer so it's saved with the model but not trained and updated during backpropagation

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

In [ ]:
## Positional Encoding test
pos_enc = PositionalEncoding(d_model, max_seq_len, dropout)

pe_output = pos_enc(output)

print("Input shape :", output.shape)
print("Output shape:", pe_output.shape)
print()
print("Before PE:", output[0][0][:5])
print("After PE :", pe_output[0][0][:5])